# Task 1: rating prediction (MSE ↓, 0-ratings skipped)

**Shipped:** bias model α + β_u + β_i (ALS), `TASK1_BIAS_PARAMS` = reg_u 1, reg_i 5, 25 epochs,
trained on nonzero rows. **Val 0.5941 | public 0.5917 | tier 2.5** (3.0 ≤ 0.58, 3.5 ≤ 0.56).

**Data facts** (checks below): no test timestamps; 0% of test pairs in train; 1.48% zero rows
(23.7% of users have one); 1.3% of test rows from single-rating users.

**Open question:** leaderboard cluster at ~0.52 (tier 4.0); cause unknown. Ask a TA.

In [21]:
%load_ext autoreload
%autoreload 2
 
import itertools

import numpy as np
import pandas as pd
from surprise import SVD, Dataset, Reader
from tqdm.auto import tqdm

from assignment_1.modules.models import fit_bias, predict_bias
from assignment_1.modules.util import (
    calc_mse,
    drop_zero_ratings,
    load_csv,
    random_split,
)

DATA_DIR = "../data"  # notebook runs from notebooks/

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
TASK_TRAIN_FILE = "tasks_1_2_train.csv"
TASK_TEST_FILE = "task_1_test.csv"
train = load_csv(TASK_TRAIN_FILE, DATA_DIR)
test = load_csv(TASK_TEST_FILE, DATA_DIR)

SCALE = (1,5)

## Data checks

In [23]:
print(f"{len(train):,} rows | {train.user.nunique():,} users | {train.item.nunique():,} items")
print(f"test unseen: users {(~test.user.isin(train.user)).mean():.1%}, items {(~test.item.isin(train.item)).mean():.1%}")
print(f"test columns: {test.columns.tolist()}  (rating column is empty: the prediction slot)")
 
key = lambda d: d["user"] + "|" + d["item"]
z = train["rating"] == 0
g = train[~z].groupby("user")["rating"].nunique()
print(f"test pairs in train: {key(test).isin(set(key(train))).mean():.2%}")
print(f"zero rows: {z.mean():.2%} | users with any zero: {train.loc[z, 'user'].nunique() / train.user.nunique():.1%}")
print(f"test rows from single-rating users: {(test.user.map(g) == 1).mean():.1%}")
train.rating.value_counts().sort_index()

1,861,643 rows | 30,909 users | 51,007 items
test unseen: users 0.0%, items 0.0%
test columns: ['user', 'item', 'rating']  (rating column is empty: the prediction slot)
test pairs in train: 0.00%
zero rows: 1.48% | users with any zero: 23.7%
test rows from single-rating users: 1.3%


rating
0     27626
1     36443
2     97748
3    314624
4    641182
5    744020
Name: count, dtype: int64

## Validation split + yardstick
One split for every experiment. It must reproduce the shipped number (0.5941) or the
comparisons below are meaningless.

In [24]:
tr, val = random_split(train, val_frac=0.25, seed=0)   # the split that reproduces 0.5941
tr, val = drop_zero_ratings(tr), drop_zero_ratings(val)
y_val = val["rating"].to_numpy()
 
bias_fit = fit_bias(tr, reg_u=1.0, reg_i=5.0, n_epochs=25, rating_scale=SCALE)
base_pred = predict_bias(bias_fit, val.user, val.item)
BASE = calc_mse(y_val, base_pred)
print(f"shipped bias model: {BASE:.4f}  (expect 0.5941)")

shipped bias model: 0.5941  (expect 0.5941)


## Helpers

In [25]:
ts = Dataset.load_from_df(tr[["user", "item", "rating"]], Reader(rating_scale=SCALE)).build_full_trainset()
val_tuples = list(val[["user", "item", "rating"]].itertuples(index=False, name=None))
 
 
def svd_mse(**params) -> float:
    m = SVD(biased=True, random_state=0, **params).fit(ts)
    return calc_mse(y_val, [p.est for p in m.test(val_tuples)])
 
 
def run_grid(score, grid: dict) -> pd.DataFrame:
    """score(**params) -> MSE for every combination in grid; sorted, with % vs the shipped model."""
    rows = []
    for values in tqdm(list(itertools.product(*grid.values()))):
        params = dict(zip(grid, values))
        mse = score(**params)
        rows.append({**params, "val_mse": mse, "vs_base_%": (1 - mse / BASE) * 100})
    return pd.DataFrame(rows).sort_values("val_mse").reset_index(drop=True)
 
 
def svd_sep(k, reg_b, reg_f, lr=0.005, epochs=20):
    """SVD with separate bias / factor regularization (SGD scale)."""
    return svd_mse(n_factors=k, reg_bu=reg_b, reg_bi=reg_b, reg_pu=reg_f, reg_qi=reg_f,
                   lr_all=lr, n_epochs=epochs)
 
 
def bias_mse(reg_u, reg_i, n_epochs=25):
    fit = fit_bias(tr, reg_u=reg_u, reg_i=reg_i, n_epochs=n_epochs, rating_scale=SCALE)
    return calc_mse(y_val, predict_bias(fit, val.user, val.item))

## E1. Bias model λ (shipped)
**Result:** best reg_u 1, reg_i 5, 25 epochs → 0.5941. More epochs: no change.

In [26]:
run_grid(
    bias_mse,
    {
        "reg_u": [0.3, 1, 3],
        "reg_i": [2, 5, 10],
        "n_epochs": [25]
    }
)

  0%|          | 0/9 [00:00<?, ?it/s]

,reg_u,reg_i,n_epochs,val_mse,vs_base_%
0,1.0,5,25,0.594096,0.000000
1,0.3,5,25,0.594209,-0.019014
2,1.0,2,25,0.594399,-0.051014
3,0.3,2,25,0.594492,-0.066595
4,3.0,5,25,0.596277,-0.367026
5,3.0,2,25,0.596641,-0.428356
6,1.0,10,25,0.596690,-0.436561
7,0.3,10,25,0.596878,-0.468202
8,3.0,10,25,0.598680,-0.771609


## E2. SVD with one λ (`reg_all`)
**Result:** best k = 1, reg_all 0.05, 50 epochs → 0.5989; blending with bias did not beat bias alone.

In [27]:
run_grid(
    svd_mse,
    {
        "n_factors": [1, 10, 50],
        "reg_all": [0.02, 0.05, 0.1, 0.3],
        "n_epochs": [20, 50]
    }
)

  0%|          | 0/24 [00:00<?, ?it/s]

,n_factors,reg_all,n_epochs,val_mse,vs_base_%
0,1,0.05,50,0.598897,-0.808026
1,1,0.10,50,0.601135,-1.184716
2,10,0.10,50,0.601459,-1.239233
3,50,0.10,50,0.601822,-1.300422
4,1,0.02,50,0.602032,-1.335819
5,10,0.05,50,0.603888,-1.648081
6,1,0.02,20,0.604109,-1.685444
7,1,0.05,20,0.605419,-1.905808
8,10,0.02,20,0.606045,-2.011258
9,10,0.05,20,0.606123,-2.024300


## E3. SVD with separate bias / factor λ
**Result:** best 0.5980 (k 5, reg_b 0.005, reg_f 0.3, lr 0.003, 60 epochs). Still worse than
bias (0.5941). reg_f barely matters → factors contribute ~nothing. Best rows at grid edges
(lowest lr, most epochs, lowest reg_b): extend before concluding.

In [ ]:
run_grid(
    svd_sep,
    {
        "k": [5],
        "reg_b": [0.005, 0.02],
        "reg_f": [0.05, 0.1, 0.3],
        "lr": [0.003, 0.01],
        "epochs": [20, 60]
    }
)

## E4. Are the factors doing anything?
Same SGD settings with factors frozen (reg_f = 100) vs free. Equal → factors add nothing.

In [ ]:
pd.Series({
    "shipped bias (ALS)": BASE,
    "SVD factors frozen": svd_sep(5, 0.005, 100, lr=0.003, epochs=60),
    "SVD best (E3)": svd_sep(5, 0.005, 0.3, lr=0.003, epochs=60),
}).round(5)

  0%|          | 0/27 [00:00<?, ?it/s]

,k,reg_b,reg_f,val_mse
1,1,0.005,0.3,0.603894
2,1,0.005,1.0,0.603894
0,1,0.005,0.1,0.603894
11,5,0.005,1.0,0.603902
20,20,0.005,1.0,0.603918
10,5,0.005,0.3,0.603932
19,20,0.005,0.3,0.604017
9,5,0.005,0.1,0.604029
5,1,0.020,1.0,0.604318
4,1,0.020,0.3,0.604318


In [ ]:
err = (base_pred - y_val) ** 2
n_u = val["user"].map(tr["user"].value_counts()).fillna(0).to_numpy()
n_i = val["item"].map(tr["item"].value_counts()).fillna(0).to_numpy()
u_std = val["user"].map(tr.groupby("user")["rating"].std()).fillna(0).to_numpy()

def err_by(x, bins, name):
    g = (pd.DataFrame({"b": pd.cut(x, bins, include_lowest=True), "e": err})
           .groupby("b", observed=True)["e"].agg(["mean", "size", "sum"]))
    g["share_of_error"] = g["sum"] / err.sum()
    print(f"\n{name}\n{g[['mean', 'size', 'share_of_error']].round(3)}")

err_by(n_u, [0, 1, 3, 10, 30, 100, 1e6], "user history size")
err_by(n_i, [0, 1, 3, 10, 30, 100, 1e6], "item popularity")
err_by(u_std, [0, 0.01, 0.5, 0.8, 1.0, 1.3, 5], "user rating std")
err_by(base_pred, [1, 3, 3.5, 4, 4.5, 5], "predicted value")
err_by(y_val, [0.5, 1.5, 2.5, 3.5, 4.5, 5.5], "true rating")

## Next
- E5: extend E3 past its edges (reg_b 0–0.005, lr 0.001–0.003, 60–120 epochs) only if E4 shows factors help.
- E6: implicit feedback (zeros as interactions): residual features first, SVD++ if they help.
- Ask a TA what reaches ~0.52 with course tools.